<a href="https://colab.research.google.com/github/ImSayvi/Sztuczna-Inteligencja/blob/main/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.protobuf
print(google.protobuf.__version__)

In [ ]:
import tensorflow_datasets as tfds
print("OK")

In [ ]:

import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds


In [ ]:
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

In [ ]:
def scale(image, label):
  image = tf.cast(image, tf.float32)
  image /= 255
  return image, label

scaled_train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

In [ ]:
from tensorflow.python.module.module import valid_identifier
BUFFER_SIZE= 1000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)
validation_data = shuffled_train_and_validation_data.take(num_validation_samples)
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

In [ ]:
BATCH_SIZE = 100
train_data = train_data.batch(BATCH_SIZE)
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)
validation_inputs, validation_targets = next(iter(validation_data))
print(validation_inputs.shape, validation_targets.shape)

In [ ]:
input_size = 784
output_size = 10
hidden_layer_size = 50

model = tf.keras.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
  tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
  tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
  tf.keras.layers.Dense(output_size, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:

NUM_EPOCHS = 10
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)
model.fit(train_data,
          epochs=NUM_EPOCHS,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose=1
)

In [ ]:


test_loss, test_accuracy = model.evaluate(test_data)
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100))

In [13]:
import cv2
import numpy as np

def preprocess_mnist_style(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # 1. Odwróć jeśli czarna cyfra na białym tle
    if np.mean(img) > 127:
        img = 255 - img

    # 2. Binaryzacja Otsu
    _, img = cv2.threshold(img, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Przytnij do bounding box + margines 4px
    coords = cv2.findNonZero(img)
    x, y, w, h = cv2.boundingRect(coords)
    img = img[y:y+h, x:x+w]
    img = cv2.copyMakeBorder(img, 4,4,4,4,
        cv2.BORDER_CONSTANT, value=0)

    # 4. Skaluj do 20x20 (jak MNIST), wstaw w 28x28
    img = cv2.resize(img, (20, 20),
        interpolation=cv2.INTER_AREA)
    canvas = np.zeros((28, 28), dtype=np.uint8)
    canvas[4:24, 4:24] = img

    # 5. Centrowanie przez środek masy
    M = cv2.moments(canvas)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        dx, dy = 14 - cx, 14 - cy
        Mat = np.float32([[1,0,dx],[0,1,dy]])
        canvas = cv2.warpAffine(canvas, Mat, (28,28))

    # 6. Normalizacja
    img_input = canvas.astype("float32") / 255.0
    img_input = img_input.reshape(1, 28, 28, 1)
    return img_input

# Użycie:
img_input = preprocess_mnist_style("4.png")
predictions = model.predict(img_input)
predicted_digit = np.argmax(predictions)
print(f"Predykcja: {predicted_digit} ({np.max(predictions)*100:.1f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Predykcja: 4 (80.8%)
